# Step 9: Error Analysis & Model Interpretability

## 1. Objective

Analyze where the Step 8 ablated XGBoost model (`models/xgboost_ablation_no_balance_ratio.joblib`, 40 features, `amount_to_sender_balance`/`amount_exceeds_sender_balance` removed) makes mistakes: false-positive and false-negative characteristics, probability separation, transaction-type/time/amount patterns, and high-confidence errors. **No model is trained, changed, or tuned in this notebook** -- this is analysis only, using the fixed 0.5 threshold established in Steps 5-8.

## 2. Model and data verification

In [ ]:
import sys
sys.path.append("..")

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.model_utils import load_split_data, exclude_features, evaluate_classifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

model = joblib.load("../models/xgboost_ablation_no_balance_ratio.joblib")

data = load_split_data()
feature_columns = data["feature_columns"]
artifact_features = ["amount_to_sender_balance", "amount_exceeds_sender_balance"]
ablated_columns = exclude_features(feature_columns, artifact_features)

print("Feature count:", len(ablated_columns))
assert len(ablated_columns) == 40
forbidden = artifact_features + ["nameOrig", "nameDest", "isFraud", "newbalanceOrig", "newbalanceDest", "isFlaggedFraud"]
for f in forbidden:
    assert f not in ablated_columns
print("Confirmed absent from feature list:", forbidden)

with open("../data/processed/split_metadata.json") as f:
    split_meta = json.load(f)
assert split_meta["train_step_range"] == [1, 323]
assert split_meta["validation_step_range"] == [324, 378]
assert split_meta["test_step_range"] == [379, 743]
print("Split ranges verified unchanged: train [1,323], validation [324,378], test [379,743]")

## 3. Prediction generation

The processed parquet stores transaction type as one-hot columns (no raw `type` string), so a `type` column is reconstructed here purely for readability in this analysis -- it is not used as a model input (the model already saw the one-hot columns during Step 8 training).

In [ ]:
val_raw = pd.read_parquet("../data/processed/validation.parquet")
test_raw = pd.read_parquet("../data/processed/test.parquet")

type_cols = ["type_CASH_IN", "type_CASH_OUT", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"]
for raw in [val_raw, test_raw]:
    raw["type"] = raw[type_cols].idxmax(axis=1).str.replace("type_", "", regex=False)

proba_val = model.predict_proba(val_raw[ablated_columns])[:, 1]
proba_test = model.predict_proba(test_raw[ablated_columns])[:, 1]

display_only = ["step", "type", "isFraud"]  # amount/oldbalanceOrg/oldbalanceDest already in ablated_columns
col_order = display_only + [c for c in ablated_columns if c not in display_only]

analysis_val = val_raw[col_order].copy()
analysis_val["predicted_probability"] = proba_val
analysis_val["predicted_class"] = (proba_val >= 0.5).astype(int)

analysis_test = test_raw[col_order].copy()
analysis_test["predicted_probability"] = proba_test
analysis_test["predicted_class"] = (proba_test >= 0.5).astype(int)

assert not analysis_val.columns.duplicated().any()
assert not analysis_test.columns.duplicated().any()
print("Validation predictions:", len(analysis_val), " predicted fraud:", int(analysis_val['predicted_class'].sum()))
print("Test predictions:", len(analysis_test), " predicted fraud:", int(analysis_test['predicted_class'].sum()))

## 4. Confusion matrix groups

In [ ]:
def confusion_groups(df):
    tn = df[(df["isFraud"] == 0) & (df["predicted_class"] == 0)]
    fp = df[(df["isFraud"] == 0) & (df["predicted_class"] == 1)]
    fn = df[(df["isFraud"] == 1) & (df["predicted_class"] == 0)]
    tp = df[(df["isFraud"] == 1) & (df["predicted_class"] == 1)]
    return tn, fp, fn, tp

groups = {}
summary_rows = []
for split_name, df in [("validation", analysis_val), ("test", analysis_test)]:
    tn, fp, fn, tp = confusion_groups(df)
    groups[split_name] = {"tn": tn, "fp": fp, "fn": fn, "tp": tp}
    n_legit, n_fraud = len(tn) + len(fp), len(fn) + len(tp)
    assert n_legit + n_fraud == len(df)
    metrics = evaluate_classifier(df["isFraud"], df["predicted_probability"], threshold=0.5)
    print(f"{split_name}: TN={len(tn):,} FP={len(fp):,} FN={len(fn):,} TP={len(tp):,}  "
          f"FPR={len(fp)/n_legit:.6f}  FNR={len(fn)/n_fraud:.6f}  "
          f"precision={metrics['precision']:.4f}  recall={metrics['recall']:.4f}  f1={metrics['f1']:.4f}")
    summary_rows.append({
        "split": split_name, "tn": len(tn), "fp": len(fp), "fn": len(fn), "tp": len(tp),
        "n_legit": n_legit, "n_fraud": n_fraud,
        "false_positive_rate": len(fp) / n_legit, "false_negative_rate": len(fn) / n_fraud,
        "precision": metrics["precision"], "recall": metrics["recall"], "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"], "pr_auc": metrics["pr_auc"],
    })

pd.DataFrame(summary_rows).to_csv("../results/error_analysis/error_summary.csv", index=False)
for split_name in ["validation", "test"]:
    groups[split_name]["fp"].to_csv(f"../results/error_analysis/{split_name}_false_positives.csv", index=False)
    groups[split_name]["fn"].to_csv(f"../results/error_analysis/{split_name}_false_negatives.csv", index=False)
print("\nSaved error_summary.csv and the 4 raw FP/FN CSVs.")

**False negatives outnumber false positives by a wide margin in both splits** (validation: 174 FN vs 32 FP; test: 991 FN vs 183 FP) -- missing real fraud is a far more common error type than misflagging a legitimate transaction, at this fixed 0.5 threshold.

## 5. False positive analysis (vs. True Negatives)

In [ ]:
numeric_features = [
    "amount", "oldbalanceOrg", "oldbalanceDest", "step", "hour_of_day", "simulated_hour", "simulated_day",
    "prior_sender_transaction_count", "prior_sender_total_amount", "prior_sender_mean_amount",
    "prior_sender_time_since_last_transaction", "prior_sender_transfer_count", "prior_sender_cashout_count",
    "prior_sender_distinct_destinations", "prior_receiver_transaction_count", "prior_receiver_total_amount",
    "prior_receiver_mean_amount", "prior_receiver_time_since_last_transaction", "prior_receiver_distinct_senders",
    "sender_transactions_previous_1_step", "sender_transactions_previous_6_steps",
    "sender_transactions_previous_24_steps", "sender_amount_previous_24_steps",
    "receiver_transactions_previous_24_steps",
]
binary_features = ["sender_balance_zero", "destination_balance_zero"]

def numeric_summary(df, cols):
    return df[cols].agg(["count", "mean", "median", "std", "min", "max"]).T

fp_num = numeric_summary(groups["test"]["fp"], numeric_features)
tn_num = numeric_summary(groups["test"]["tn"], numeric_features)
cmp_fp_tn = fp_num.join(tn_num, lsuffix="_FP", rsuffix="_TN")
print("=== TEST: False Positives (n=183) vs True Negatives (n=914,428) ===")
cmp_fp_tn[["mean_FP", "median_FP", "mean_TN", "median_TN"]]

In [ ]:
print("FP type distribution:")
print(groups["test"]["fp"]["type"].value_counts(normalize=True).to_string())
print("\nTN type distribution:")
print(groups["test"]["tn"]["type"].value_counts(normalize=True).to_string())
print("\nBinary flags (mean = proportion of 1s):")
for col in binary_features:
    print(f"  {col}: FP={groups['test']['fp'][col].mean():.4f}  TN={groups['test']['tn'][col].mean():.4f}")

**False positives were concentrated in TRANSFER (29.5%) and CASH_OUT (70.5%) transactions exclusively** -- exactly the two types fraud occurs in, unsurprising since the model never predicts fraud outside them. False positives had `sender_balance_zero = 0` in 100% of cases (vs. 31.7% for true negatives) and noticeably lower `oldbalanceOrg` on average (170K vs. 786K) -- consistent with false positives resembling the risk profile of genuine fraud (moderate sender balance, TRANSFER/CASH_OUT type) rather than a random slice of legitimate traffic. This is reported as an observed concentration, not a causal claim.

## 6. False negative analysis (vs. True Positives)

In [ ]:
fn_num = numeric_summary(groups["test"]["fn"], numeric_features)
tp_num = numeric_summary(groups["test"]["tp"], numeric_features)
cmp_fn_tp = fn_num.join(tp_num, lsuffix="_FN", rsuffix="_TP")
print("=== TEST: False Negatives (n=991) vs True Positives (n=3,015) ===")
cmp_fn_tp[["mean_FN", "median_FN", "mean_TP", "median_TP"]]

In [ ]:
print("FN type distribution:")
print(groups["test"]["fn"]["type"].value_counts(normalize=True).to_string())
print("\nTP type distribution:")
print(groups["test"]["tp"]["type"].value_counts(normalize=True).to_string())
print("\nBinary flags:")
for col in binary_features:
    print(f"  {col}: FN={groups['test']['fn'][col].mean():.4f}  TP={groups['test']['tp'][col].mean():.4f}")

**The clearest pattern: missed fraud (false negatives) has a much smaller transaction amount than caught fraud (true positives).** Median amount: 144,452 (FN) vs. 850,003 (TP) -- roughly 6x smaller. `destination_balance_zero` (the classic drained-account receiver signature) is present in only 42.6% of false negatives vs. 73.3% of true positives. Type distribution also shifts: false negatives skew toward CASH_OUT (79.9%) while true positives skew toward TRANSFER (59.8%) -- the model is comparatively better at catching TRANSFER fraud than CASH_OUT fraud. Missed fraud also involves receivers with more prior transaction history on average (4.66 vs. 2.15) -- explored further in section 11.

## 7. Probability distributions

In [ ]:
def proba_summary(df):
    p = df["predicted_probability"]
    return {"count": len(p), "mean": float(p.mean()), "median": float(p.median()), "std": float(p.std()), "min": float(p.min()), "max": float(p.max())}

prob_rows = []
for split in ["validation", "test"]:
    for label in ["tn", "fp", "fn", "tp"]:
        s = proba_summary(groups[split][label])
        prob_rows.append({"split": split, "group": label.upper(), **s})
        print(f"{split} {label.upper()}: {s}")

prob_summary_df = pd.DataFrame(prob_rows)
prob_summary_df.to_csv("../results/error_analysis/probability_group_summary.csv", index=False)
print("\nSaved results/error_analysis/probability_group_summary.csv")

In [ ]:
def plot_distribution(groups, split, path):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    colors = {"tn": "#4C72B0", "fp": "#DD8452", "fn": "#C44E52", "tp": "#55A868"}
    labels = {"tn": "True Negative", "fp": "False Positive", "fn": "False Negative", "tp": "True Positive"}
    for key in ["tn", "fp", "fn", "tp"]:
        p = groups[split][key]["predicted_probability"]
        weight = np.ones_like(p) / max(len(p), 1)
        ax.hist(p, bins=40, range=(0, 1), alpha=0.55, label=f"{labels[key]} (n={len(p):,})", color=colors[key], weights=weight)
    ax.set_xlabel("Predicted fraud probability")
    ax.set_ylabel("Proportion within group")
    ax.set_title(f"Predicted Probability Distribution by Confusion-Matrix Group ({split.capitalize()})")
    ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="Threshold = 0.5")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()

plot_distribution(groups, "test", "../figures/error_analysis/01_probability_distribution_test.png")
plot_distribution(groups, "validation", "../figures/error_analysis/02_probability_distribution_validation.png")

**The model shows clear separation on average** -- true negatives cluster near 0 (test median ~4.4e-7) and true positives cluster near 1 (test median ~0.993). But the **false negative distribution is skewed toward very low probabilities, not clustered near the 0.5 boundary** (test median 0.034) -- most missed fraud is confidently misclassified, not a borderline near-miss. False positives sit at moderate-to-high confidence (test median 0.75), meaning most are not marginal calls either.

## 8. Transaction-type analysis

In [ ]:
def by_type_table(df, split_name):
    rows = []
    for t in ["TRANSFER", "CASH_OUT", "PAYMENT", "CASH_IN", "DEBIT"]:
        sub = df[df["type"] == t]
        total = len(sub)
        fraud_count = int(sub["isFraud"].sum())
        pred_fraud_count = int(sub["predicted_class"].sum())
        tp = int(((sub["isFraud"] == 1) & (sub["predicted_class"] == 1)).sum())
        fp = int(((sub["isFraud"] == 0) & (sub["predicted_class"] == 1)).sum())
        fn = int(((sub["isFraud"] == 1) & (sub["predicted_class"] == 0)).sum())
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        rows.append({"split": split_name, "type": t, "total_transactions": total, "fraud_count": fraud_count,
                     "predicted_fraud_count": pred_fraud_count, "true_positive": tp, "false_positive": fp,
                     "false_negative": fn, "precision": precision, "recall": recall})
    return pd.DataFrame(rows)

val_by_type = by_type_table(analysis_val, "validation")
test_by_type = by_type_table(analysis_test, "test")
pd.concat([val_by_type, test_by_type], ignore_index=True).to_csv("../results/error_analysis/error_by_transaction_type.csv", index=False)
print("=== Validation ===")
print(val_by_type.to_string(index=False))
print("\n=== Test ===")
print(test_by_type.to_string(index=False))

**PAYMENT, CASH_IN, and DEBIT have zero fraud and zero predicted fraud in both splits** -- the model never predicts fraud outside TRANSFER/CASH_OUT, consistent with the training distribution (Step 2). **Recall is substantially higher for TRANSFER (84.0% validation, 90.1% test) than CASH_OUT (54.3% validation, 60.5% test)** in both splits -- CASH_OUT fraud is comparatively harder for this model to catch.

## 9. Simulated-time analysis

`step`/`simulated_day`/`hour_of_day` are PaySim SIMULATED time, not real calendar time -- results here describe this specific 31-day simulation, not a real-world time-of-day or day-of-week effect.

In [ ]:
def by_day_table(df, split_name):
    d = df.copy()
    d["fp"] = ((d["isFraud"] == 0) & (d["predicted_class"] == 1)).astype(int)
    d["fn"] = ((d["isFraud"] == 1) & (d["predicted_class"] == 0)).astype(int)
    agg = d.groupby("simulated_day").agg(
        total_transactions=("isFraud", "size"), fraud_count=("isFraud", "sum"),
        false_positives=("fp", "sum"), false_negatives=("fn", "sum"),
    ).reset_index()
    agg["split"] = split_name
    return agg

val_by_day = by_day_table(analysis_val, "validation")
test_by_day = by_day_table(analysis_test, "test")
pd.concat([val_by_day, test_by_day], ignore_index=True).to_csv("../results/error_analysis/error_by_simulated_time.csv", index=False)
print("=== Validation by PaySim simulated day ===")
print(val_by_day.to_string(index=False))
print("\n=== Test by PaySim simulated day ===")
print(test_by_day.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, tbl, split_label, step_range in [
    (axes[0], val_by_day, "Validation", "steps 324-378"),
    (axes[1], test_by_day, "Test", "steps 379-743"),
]:
    ax.bar(tbl["simulated_day"], tbl["false_positives"], color="#DD8452", label="False Positives")
    ax.bar(tbl["simulated_day"], tbl["false_negatives"], bottom=tbl["false_positives"], color="#C44E52", label="False Negatives")
    ax.set_title(f"Error Counts by PaySim Simulated Day -- {split_label} ({step_range})")
    ax.set_xlabel("PaySim simulated day (NOT real calendar time)")
    ax.set_ylabel("Error count")
    ax.legend()
plt.tight_layout()
plt.savefig("../figures/error_analysis/03_error_rate_over_time.png", dpi=150)
plt.show()

**Simulated days 17, 25, and 27 (test) show a concentration of false positives (46, 28, and 29 respectively)** relative to neighboring days -- these specific periods and counts are reported as observed; no explanation is inferred beyond what the data shows. False negatives are present across nearly every simulated day in both splits, without an obvious single concentrated period -- missed fraud is spread out rather than isolated to one time window.

## 10. Amount-based analysis

Bins use quantiles computed independently within each split (validation bins from validation data only, test bins from test data only) -- descriptive only, never used to modify the model.

In [ ]:
quantile_edges = [0, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
bin_labels = ["0-25pct", "25-50pct", "50-75pct", "75-90pct", "90-95pct", "95-99pct", "99-100pct"]

def amount_bin_table(df, split_name):
    d = df.copy()
    edges = d["amount"].quantile(quantile_edges).values
    edges[0] = -0.01
    edges[-1] = edges[-1] + 1
    d["amount_bin"] = pd.cut(d["amount"], bins=edges, labels=bin_labels, include_lowest=True, duplicates="drop")
    rows = []
    for b in d["amount_bin"].cat.categories:
        sub = d[d["amount_bin"] == b]
        tp = int(((sub["isFraud"] == 1) & (sub["predicted_class"] == 1)).sum())
        fp = int(((sub["isFraud"] == 0) & (sub["predicted_class"] == 1)).sum())
        fn = int(((sub["isFraud"] == 1) & (sub["predicted_class"] == 0)).sum())
        precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        rows.append({"split": split_name, "amount_bin": b, "total_transactions": len(sub),
                     "fraud_count": int(sub["isFraud"].sum()), "predicted_fraud_count": int(sub["predicted_class"].sum()),
                     "false_positives": fp, "false_negatives": fn, "precision": precision, "recall": recall})
    return pd.DataFrame(rows)

val_amt = amount_bin_table(analysis_val, "validation")
test_amt = amount_bin_table(analysis_test, "test")
pd.concat([val_amt, test_amt], ignore_index=True).to_csv("../results/error_analysis/error_by_amount_bin.csv", index=False)
print("=== Validation by amount bin ===")
print(val_amt.to_string(index=False))
print("\n=== Test by amount bin ===")
print(test_amt.to_string(index=False))

**Recall rises with transaction amount in both splits.** Test recall: 34.2% (0-25th pct) -> 50.0% -> 61.6% -> 67.9% -> 48.9% -> 91.4% -> **100%** (top 1%). Validation shows the same overall pattern (16.7% -> 100%). This directly confirms and quantifies section 6's finding: the model is much more likely to miss smaller-value fraud than large-value fraud. The 90-95th percentile bin dips slightly below the 75-90th in both splits -- reported as observed, not explained further.

## 11. High-confidence errors

In [ ]:
display_cols = ["step", "type", "amount", "oldbalanceOrg", "oldbalanceDest", "predicted_probability", "isFraud",
                 "sender_balance_zero", "destination_balance_zero", "prior_sender_transaction_count",
                 "prior_receiver_transaction_count"]

fp_all = pd.concat([groups["validation"]["fp"].assign(split="validation"), groups["test"]["fp"].assign(split="test")], ignore_index=True)
fn_all = pd.concat([groups["validation"]["fn"].assign(split="validation"), groups["test"]["fn"].assign(split="test")], ignore_index=True)

top_fp = fp_all.sort_values("predicted_probability", ascending=False).head(20)
top_fn = fn_all.sort_values("predicted_probability", ascending=True).head(20)

print("=== TOP 20 highest-probability FALSE POSITIVES ===")
print(top_fp[["split"] + display_cols].to_string(index=False))
print("\n=== TOP 20 lowest-probability FALSE NEGATIVES ===")
print(top_fn[["split"] + display_cols].to_string(index=False))

high_conf = pd.concat([
    top_fp[["split"] + display_cols].assign(error_type="false_positive"),
    top_fn[["split"] + display_cols].assign(error_type="false_negative"),
], ignore_index=True)
high_conf.to_csv("../results/error_analysis/high_confidence_errors.csv", index=False)
print("\nSaved results/error_analysis/high_confidence_errors.csv")

**High-confidence false positives** are almost all TRANSFER/CASH_OUT transactions where `destination_balance_zero=1` OR the receiver has substantial prior transaction history and balance -- i.e., legitimate transactions that happen to share surface characteristics with the fraud pattern.

**High-confidence (lowest-probability) false negatives are notably uniform: all 20 are CASH_OUT**, and most involve a receiver with meaningful prior transaction history (several with 20-42 prior transactions). This is a real, visible pattern in the data: fraud routed through receivers that already look like established, high-activity accounts is where this model is most confidently wrong -- not a borderline miss, but a strongly confident misclassification.

## 12. XGBoost feature importance (from Step 8, not recomputed)

In [ ]:
imp = pd.read_csv("../results/artifact_ablation_feature_importance_xgb.csv")
top15 = imp.sort_values("importance", ascending=False).head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top15["feature"], top15["importance"], color="#4C72B0")
ax.set_xlabel("Gain-based importance (from Step 8 XGBoost ablated model)")
ax.set_title("Top 15 XGBoost Feature Importances\n(40-feature ablated model -- gain-based, not causal)")
plt.tight_layout()
plt.savefig("../figures/error_analysis/04_xgboost_feature_importance.png", dpi=150)
plt.show()

print(top15.iloc[::-1].to_string(index=False))

Gain-based importance measures how useful a feature was for reducing this specific trained model's loss -- an association/usefulness measure, not causal importance. `destination_balance_zero`, `is_transfer_or_cash_out`, and `prior_receiver_has_history` lead the ablated model, consistent with sections 6 and 11's finding that receiver-side history is central to both the model's decisions and its most confident mistakes.

## 13. Findings (Key Findings summary)

- **False negatives are far more common than false positives** at threshold 0.5 (validation: 174 vs. 32; test: 991 vs. 183).
- **False positives concentrate exclusively in TRANSFER and CASH_OUT** -- the only types the model ever flags -- and resemble the fraud risk profile (lower sender balance, never zero) more than a random legitimate transaction.
- **Missed fraud has recognizable characteristics**: smaller transaction amounts (median ~144K vs. ~850K for caught fraud), less often showing the destination-zero-balance signature (42.6% vs. 73.3%), and skewed toward CASH_OUT rather than TRANSFER.
- **Model probabilities show clear average separation** (true negatives near 0, true positives near 1), but false negatives skew toward very low probabilities rather than clustering near the 0.5 boundary -- most missed fraud is confidently misclassified, not a near-miss.
- **The most important remaining features** are `destination_balance_zero`, `is_transfer_or_cash_out`, and `prior_receiver_has_history` -- receiver-side signals dominate now that the two removed balance-ratio features are gone.
- **Errors do not concentrate in one simulated period overall**, though specific days (17, 25, 27 in test) show locally elevated false-positive counts.
- **High-confidence mistakes exist and show a pattern**: the most confidently-wrong false negatives are uniformly CASH_OUT transactions to receivers with substantial prior transaction history -- within this PaySim experiment, the model assigns lower predicted fraud probabilities to transactions that resemble routing through an established-looking receiver account, which is exactly where fraud slips through most confidently.

All statements above describe what the analysis shows within this fitted model and this dataset; none assert a causal or universal fraud mechanism.

## 14. Limitations

**PaySim is synthetic.** The dataset contains simulation-specific behavior and distribution characteristics documented throughout this project (Steps 2, 4, 6-8). Therefore:

- The error patterns described above are specific to this dataset and this fitted model -- they are not claimed to generalize to other fraud datasets or systems.
- PaySim's `step`/`simulated_day`/`hour_of_day` are simulated time, not real-world transaction time; the time-based findings in section 9 describe this specific 31-day simulation only.
- Observed feature importance (section 12) reflects usefulness within this trained model, not proof of real-world fraud causality.
- This analysis, and the model it examines, should not be directly generalized to NPCI/UPI or other production payment systems. PaySim is a synthetic dataset, so these results should be interpreted as an experimental benchmark rather than evidence of production fraud-detection performance.